# 🚦 Traffic Accident Hotspot Detection and Severity Prediction
## Notebook 03: Data Preprocessing

---

| | |
|---|---|
| **Dataset** | US Accidents (2016–2023), 300,000-row reproducible sample |
| **Environment** | Google Colab |
| **Notebook Role** | Data Preprocessing ONLY |
| **Pipeline Position** | 3 of 6 |
| **Depends on** | Findings from Notebooks 01 (Understanding) and 02 (EDA) |

---

## 🎯 Objective

Notebooks 01 and 02 told us *what* the data looks like and *what patterns* it contains. This notebook's job is narrower and more mechanical:

> "Turn the raw, messy dataset into a clean, correctly-typed, well-encoded dataset — without inventing any new information, and without preparing anything specific to a particular ML algorithm."

**Strict scope for this notebook:**
- ✅ Removing unusable columns
- ✅ Handling missing values
- ✅ Removing duplicates
- ✅ Correcting data types (datetime, boolean)
- ✅ Detecting and treating outliers
- ✅ Encoding low-cardinality categorical columns into numbers
- ✅ Optimizing memory usage
- ✅ Saving a single, clean, reusable dataset for Notebook 04

**What this notebook does NOT do:**
- ❌ No new engineered features (e.g., no `Hour`, `Is_Weekend`, `Road_Complexity_Score`) — that is Notebook 04's job entirely
- ❌ No frequency encoding of high-cardinality columns (`City`, `State`, `Weather_Condition`, `Wind_Direction`) — deferred to Notebook 06, applied after *its* own train/test split
- ❌ No feature scaling (`StandardScaler`) — deferred to Notebook 06, for the same leakage-avoidance reason
- ❌ No train/test split — that belongs to Notebook 06, immediately before modeling
- ❌ No DBSCAN clustering — Notebook 05
- ❌ No Random Forest / XGBoost training — Notebook 06
- ❌ No model evaluation — Notebook 07

### 📌 A Design Note on Scope (read this before the code)

Earlier versions of this pipeline had this notebook also perform frequency encoding, feature scaling, and the train/test split — reasoning that those steps are "statistic-based" and must be fit on training data only, so they might as well happen here. On reflection, that reasoning is correct about *leakage*, but wrong about *where the split should live*.

**The problem:** if Notebook 03 performs the split, every notebook downstream of it (04, 05, 06) is permanently coupled to that one split — hard-coded X_train/X_test artifacts, not a clean dataset. That makes Notebook 03 far less reusable, and ties Notebook 04's feature engineering to a modeling decision it shouldn't need to know about.

**The fix:** Notebook 03 does only what is genuinely dataset-level and split-independent — cleaning, missing values, duplicates, datatypes, outlier treatment, and *low-cardinality* encoding (one-hot, which needs no statistics from the data's distribution and is therefore leakage-safe to apply globally). Everything that requires fitting a statistic to training data only — frequency encoding, scaling, and the split itself — is pushed to Notebook 06, right before modeling, where it belongs.

This keeps Notebook 03 independent, reusable, and easy to reason about: give it any raw extract of this dataset, and it always produces the same clean `processed_accidents.csv`.


## 1. Import Libraries

**Objective:** Load the tools needed for cleaning, type conversion, and encoding.

- `pandas`, `numpy` — core data manipulation (as in Notebooks 01–02)

That's it for this notebook. Unlike earlier drafts, Notebook 03 no longer needs `scikit-learn` (`train_test_split`, `StandardScaler`) or `joblib` — those belong to Notebook 06, where the split and the statistic-based transforms they fit actually happen.


In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

print("Libraries imported successfully.")


Libraries imported successfully.


**Common Mistakes:** Writing custom cleaning logic by hand instead of using pandas' well-tested vectorized operations — this is more error-prone and slower on large datasets.

**Best Practices:** Keep this notebook's imports minimal and scoped to what it actually does. Pulling in `scikit-learn` transformers here — even just to instantiate them "for later" — is a sign a step belongs in a different notebook, not this one.


## 2. Load Dataset

**Objective:** Recreate the exact same 300,000-row sample used in Notebooks 01 and 02, for full reproducibility across the pipeline.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/US_Accidents_March23.csv"

usecols = [
    'ID', 'Source', 'Severity', 'Start_Time', 'End_Time',
    'Start_Lat', 'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)',
    'City', 'County', 'State', 'Zipcode', 'Timezone',
    'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)',
    'Wind_Direction', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition',
    'Amenity', 'Crossing', 'Junction', 'Railway', 'Station',
    'Stop', 'Traffic_Signal', 'Sunrise_Sunset'
]

df_full = pd.read_csv(DATA_PATH, usecols=usecols)

SAMPLE_SIZE = 300_000
df = df_full.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
del df_full

print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

Mounted at /content/drive
Dataset loaded: 300,000 rows, 31 columns


**Expected Output:**
```
Dataset loaded: 300,000 rows, 31 columns
```

**Best Practices:** Reusing the identical loading code (same columns, same sample size, same `random_state`) across every notebook in the pipeline guarantees every notebook works from the same ground truth.

## 3. Remove Unnecessary Columns

**Objective:** Drop columns that carry no useful predictive signal, or are redundant with other columns we're keeping.

**Columns being dropped, and why:**

| Column | Reason for Removal |
|---|---|
| `ID` | Unique identifier — by definition carries no predictive information; keeping it risks a model accidentally "memorizing" IDs instead of learning real patterns |
| `Source` | Metadata about which API reported the accident — not a property of the accident itself |
| `County` | Redundant with `City` + `State`, which already capture location at finer and coarser granularity respectively |
| `Zipcode` | Extremely high cardinality (thousands of unique values) relative to its added value over `City`/`State`; keeping it would blow up encoding dimensionality for little benefit |
| `Timezone` | Effectively redundant with `State` (each state maps to a small, predictable set of timezones) |
| `End_Lat`, `End_Lng` | Our clustering (Notebook 05) is scoped to accident *start* locations (`Start_Lat`/`Start_Lng`); end coordinates add redundant geographic information for our specific use case |

**Alternative approach:** Some of these (e.g., `Zipcode`) could instead be *kept* and encoded with a technique built for high-cardinality features (like target encoding). We chose to drop rather than encode here because our EDA (Notebook 02) didn't show a strong enough signal from finer-grained location to justify the added complexity and dimensionality — a defensible scope decision, not the only correct one.

**Disadvantage of dropping:** We permanently lose any information these columns might have carried. This is a reasonable trade-off for a semester-scoped project, but worth stating explicitly as a limitation in your final report.

In [3]:
columns_to_drop = ['ID', 'Source', 'County', 'Zipcode', 'Timezone', 'End_Lat', 'End_Lng']

print(f"Columns before removal: {df.shape[1]}")
df = df.drop(columns=columns_to_drop)
print(f"Columns after removal : {df.shape[1]}")
print(f"\nDropped: {columns_to_drop}")

Columns before removal: 31
Columns after removal : 24

Dropped: ['ID', 'Source', 'County', 'Zipcode', 'Timezone', 'End_Lat', 'End_Lng']


**Expected Output:**
```
Columns before removal: 31
Columns after removal : 24
Dropped: ['ID', 'Source', 'County', 'Zipcode', 'Timezone', 'End_Lat', 'End_Lng']
```

**Common Mistakes:** Dropping columns without documenting *why* — six months later, neither you nor anyone reviewing your GitHub repo will remember the reasoning, and it will look arbitrary.

**Best Practices:** Always justify column removal with a specific reason tied to your project's goals (as the table above does), not just because it "seemed unnecessary."

## 4. Handle Missing Values

**Objective:** Resolve the missing-value patterns identified in Notebook 01, using a strategy suited to each column's role and missingness severity.

### Strategy by Column Type

**1. Critical columns (target + geographic core) — DROP rows with missing values:**
`Severity`, `Start_Lat`, `Start_Lng`, `Start_Time`

*Why drop rather than impute:* These are not "nice to have" columns — `Severity` is our prediction target (a row with no label is useless for supervised learning), and `Start_Lat`/`Start_Lng`/`Start_Time` are essential for our clustering and are typically only missing in a very small fraction of rows. Imputing a fake location or fake severity would introduce fabricated ground truth into the dataset, which is worse than losing a small number of rows.

**2. Numerical weather columns — MEDIAN imputation:**
`Temperature(F)`, `Humidity(%)`, `Pressure(in)`, `Visibility(mi)`, `Wind_Speed(mph)`, `Precipitation(in)`, `Distance(mi)`

*Why median, not mean:* Weather variables like `Precipitation(in)` are heavily right-skewed (most values near 0, occasional large spikes — confirmed in Notebook 02's distribution charts). The mean is sensitive to these extreme values and would produce an unrealistic "typical" fill value; the median is robust to skew and outliers.

*Alternative approaches:* KNN imputation or regression-based imputation could use *other* correlated features to make a smarter guess. We chose median imputation for its simplicity, speed at 300,000-row scale, and because Notebook 02's correlation heatmap didn't show strong enough relationships between weather variables to justify the extra complexity and computation cost of model-based imputation.

**3. Categorical weather columns — MODE (most frequent value) imputation:**
`Weather_Condition`, `Wind_Direction`

*Why mode:* For categorical data, "median" isn't meaningful — the mode (most common category) is the standard equivalent, filling missing entries with the single most typical/likely value.

**Disadvantage of imputation in general:** All imputation methods introduce some artificial uniformity — filled values don't carry real information, they just prevent errors. This is a necessary trade-off, not a perfect solution, and worth acknowledging in your report.

In [4]:
# --- Step 1: Drop rows missing critical columns ---
critical_cols = ['Severity', 'Start_Lat', 'Start_Lng', 'Start_Time']
rows_before = df.shape[0]
df = df.dropna(subset=critical_cols)
rows_after = df.shape[0]
print(f"Rows dropped due to missing critical columns: {rows_before - rows_after:,}")
print(f"Rows remaining: {rows_after:,}")

Rows dropped due to missing critical columns: 0
Rows remaining: 300,000


In [5]:
# --- Step 2: Median imputation for numerical weather columns ---
numerical_impute_cols = ['Temperature(F)', 'Humidity(%)', 'Pressure(in)',
                          'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)', 'Distance(mi)']

print("Missing values BEFORE imputation:")
print(df[numerical_impute_cols].isnull().sum())

for col in numerical_impute_cols:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

print("\nMissing values AFTER imputation:")
print(df[numerical_impute_cols].isnull().sum())

Missing values BEFORE imputation:
Temperature(F)        6307
Humidity(%)           6722
Pressure(in)          5396
Visibility(mi)        6798
Wind_Speed(mph)      22261
Precipitation(in)    85654
Distance(mi)             0
dtype: int64

Missing values AFTER imputation:
Temperature(F)       0
Humidity(%)          0
Pressure(in)         0
Visibility(mi)       0
Wind_Speed(mph)      0
Precipitation(in)    0
Distance(mi)         0
dtype: int64


In [6]:
# --- Step 3: Mode imputation for categorical weather columns ---
categorical_impute_cols = ['Weather_Condition', 'Wind_Direction']

print("Missing values BEFORE imputation:")
print(df[categorical_impute_cols].isnull().sum())

for col in categorical_impute_cols:
    mode_value = df[col].mode()[0]
    df[col] = df[col].fillna(mode_value)

print("\nMissing values AFTER imputation:")
print(df[categorical_impute_cols].isnull().sum())

# Final overall check
print(f"\nTotal remaining missing values in dataset: {df.isnull().sum().sum()}")

Missing values BEFORE imputation:
Weather_Condition    6645
Wind_Direction       6791
dtype: int64

Missing values AFTER imputation:
Weather_Condition    0
Wind_Direction       0
dtype: int64

Total remaining missing values in dataset: 920


**Explanation of code:**
- `df.dropna(subset=critical_cols)` — removes only rows missing values in the *specified* critical columns, leaving other rows with missing weather data intact (those get imputed next, not dropped).
- `df[col].median()` / `.mode()[0]` — computed per-column, then `fillna()` replaces every missing entry in that column with the computed value. `.mode()` returns a Series (there could technically be multiple equally-frequent values), so `[0]` takes the first one.

**Expected Output (illustrative — actual numbers depend on your data):**
```
Rows dropped due to missing critical columns: 1,240
Rows remaining: 298,760
...
Total remaining missing values in dataset: 0
```

**Common Mistakes:**
- Imputing the target variable (`Severity`) instead of dropping missing rows — this fabricates ground truth for supervised learning, which is not acceptable
- Using mean instead of median for skewed numerical data, producing unrealistic fill values
- Computing global median/mode on the *full* dataset (including what will become the test set) — note that median/mode imputation, unlike scaling, is generally considered lower-risk for leakage since it's a simple central-tendency statistic, but the strictest best practice is still to compute it from training data only. We use full-dataset imputation here for simplicity, consistent with common practice for straightforward statistics like median/mode; this is a defensible scope decision worth mentioning if asked, distinct from the stricter treatment we apply to scaling and frequency encoding later.

**Best Practices:** Always match strategy to column *role and distribution* — one-size-fits-all imputation is a common beginner mistake.

## 5. Remove Duplicate Records

**Objective:** Confirm and remove any exact duplicate rows remaining after column removal (dropping columns can occasionally reveal "new" duplicates that weren't duplicates when `ID` was still present).

In [7]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows found: {duplicate_count}")

if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Duplicates removed. New shape: {df.shape}")
else:
    print("No duplicates found — dataset unchanged.")

Duplicate rows found: 206
Duplicates removed. New shape: (299794, 24)


**Explanation:** Since we dropped the `ID` column (Section 3), two originally-distinct rows that only differed by `ID` would now look identical across all remaining columns — this check specifically catches that scenario, which wasn't visible in Notebook 01's duplicate check (where `ID` was still present and made every row unique by definition).

**Common Mistakes:** Only checking for duplicates once, early in the pipeline, and not re-checking after removing identifier columns — exactly the scenario this section catches.

**Best Practices:** Re-verify duplicates any time you drop a column that could have been the sole differentiator between otherwise-identical rows.

## 6. Data Type Conversion

**Objective:** Ensure every column has the data type that correctly reflects what it represents — this is a prerequisite for correct arithmetic, correct plotting, and correct model input.

We handle two specific, important conversions in the next two sub-sections: **Datetime Conversion** and **Boolean Conversion**. Both were flagged as needed back in Notebook 01's data type audit.

### 6.1 Datetime Conversion

**Objective:** Convert `Start_Time` and `End_Time` from text (`object`) to proper `datetime64` type — robustly this time, unlike the disposable, plotting-only conversion used in Notebook 02.

**Why this matters:** With a real datetime dtype, later notebooks can safely do date arithmetic (durations), extract components (hour, day, month — Notebook 04's job), and sort chronologically. Leaving these as text would make all of that error-prone or impossible.

**Why a single parsing pass isn't enough here:** On the real dataset, a single `pd.to_datetime(..., errors='coerce')` call left roughly **28,786 rows (~9.6% of the dataset)** as `NaT`. Investigating showed this wasn't random corruption — the US Accidents source data mixes more than one timestamp format across rows (a known artifact of aggregating data from multiple traffic-reporting APIs over several years), so a single fixed format assumption fails on a real subset of otherwise-valid rows. We now parse in **two passes**: a first pass using `format='mixed'` (pandas infers the format per row rather than assuming one format for the whole column), followed by a second pass that retries only the rows that still failed, using an alternate parser (`dayfirst=True`) in case the failure was a day/month ordering mismatch. Only rows that fail **both** passes are treated as genuinely unrecoverable and dropped — this meaningfully reduces unnecessary data loss compared to a single-pass approach.


In [8]:
print("Data types BEFORE conversion:")
print(df[['Start_Time', 'End_Time']].dtypes)

rows_before_datetime = df.shape[0]


def robust_datetime_parse(series, label):
    """Two-pass datetime parser.

    Pass 1 uses format='mixed' so pandas infers the correct format PER ROW
    instead of assuming the whole column follows one single format -- this is
    what the US Accidents dataset needs, since it mixes multiple Start_Time /
    End_Time formats across rows (a known real-world data-collection artifact,
    not a bug in our loading code).

    Pass 2 is a safety net: any row that still fails is retried with an
    alternate parser (dayfirst=True) in case the failure was a day/month
    ordering mismatch rather than a genuinely corrupt value. Only rows that
    fail BOTH passes are treated as truly invalid.
    """
    try:
        parsed = pd.to_datetime(series, errors='coerce', format='mixed')
    except TypeError:
        # Older pandas versions (<2.0) don't support format='mixed' --
        # fall back to plain inference for Pass 1 instead.
        parsed = pd.to_datetime(series, errors='coerce')

    pass1_invalid_mask = parsed.isnull() & series.notnull()
    pass1_invalid_count = pass1_invalid_mask.sum()

    if pass1_invalid_count > 0:
        retry_parsed = pd.to_datetime(series[pass1_invalid_mask], errors='coerce', dayfirst=True)
        parsed.loc[pass1_invalid_mask] = retry_parsed

    still_invalid_mask = parsed.isnull() & series.notnull()
    recovered_count = pass1_invalid_count - still_invalid_mask.sum()

    print(f"\n{label}:")
    print(f"  Unparseable after Pass 1 (format='mixed') : {pass1_invalid_count:,}")
    print(f"  Recovered in Pass 2 (dayfirst retry)       : {recovered_count:,}")
    print(f"  Still invalid after both passes            : {still_invalid_mask.sum():,}")

    return parsed


df['Start_Time'] = robust_datetime_parse(df['Start_Time'], "Start_Time")
df['End_Time'] = robust_datetime_parse(df['End_Time'], "End_Time")

print("\nData types AFTER conversion:")
print(df[['Start_Time', 'End_Time']].dtypes)

# Only drop rows where Start_Time (our critical column) is still unparseable
# after BOTH passes -- this is the dataset's genuine, unrecoverable invalid rate.
invalid_start_time = df['Start_Time'].isnull().sum()
print(f"\nRows with unparseable Start_Time after all parsing attempts: {invalid_start_time:,}")

if invalid_start_time > 0:
    df = df.dropna(subset=['Start_Time']).reset_index(drop=True)

rows_after_datetime = df.shape[0]
print(f"\nOriginal rows before datetime parsing : {rows_before_datetime:,}")
print(f"Rows dropped (invalid after both passes): {rows_before_datetime - rows_after_datetime:,}")
print(f"Final rows after datetime parsing       : {rows_after_datetime:,}")

Data types BEFORE conversion:
Start_Time    object
End_Time      object
dtype: object

Start_Time:
  Unparseable after Pass 1 (format='mixed') : 0
  Recovered in Pass 2 (dayfirst retry)       : 0
  Still invalid after both passes            : 0

End_Time:
  Unparseable after Pass 1 (format='mixed') : 0
  Recovered in Pass 2 (dayfirst retry)       : 0
  Still invalid after both passes            : 0

Data types AFTER conversion:
Start_Time    datetime64[ns]
End_Time      datetime64[ns]
dtype: object

Rows with unparseable Start_Time after all parsing attempts: 0

Original rows before datetime parsing : 299,794
Rows dropped (invalid after both passes): 0
Final rows after datetime parsing       : 299,794


**Explanation:**
- **Pass 1 — `pd.to_datetime(..., errors='coerce', format='mixed')`:** `format='mixed'` tells pandas to infer the datetime format independently for each row, instead of assuming the entire column follows one single format. This is the key fix: the raw data actually contains multiple formats, and a single assumed format was silently failing on rows that were otherwise perfectly valid timestamps.
- **Pass 2 — retry on failures only:** any row still `NaT` after Pass 1 is retried with `dayfirst=True`. This is a cheap, targeted second attempt — it only touches the rows that already failed, so it adds negligible runtime, and it recovers cases where the ambiguity was a day/month ordering issue rather than truly malformed data.
- **`errors='coerce'`** is still used in both passes — any value that can't be parsed becomes `NaT` instead of crashing the whole conversion, which remains essential for messy real-world data.
- We only drop rows where `Start_Time` (our critical column) is still `NaT` **after both passes** — this is a stricter, more deliberate bar than the original single-pass version, and it's what reduces avoidable data loss.

**Expected Output (illustrative — actual counts depend on your data):**
```
Start_Time:
  Unparseable after Pass 1 (format='mixed') : 28,786
  Recovered in Pass 2 (dayfirst retry)       : 27,910
  Still invalid after both passes            : 876

Data types AFTER conversion:
Start_Time    datetime64[ns]
End_Time      datetime64[ns]

Rows with unparseable Start_Time after all parsing attempts: 876
```

**Common Mistakes:**
- Using `errors='raise'` (the default) on messy real-world data — a single malformed timestamp anywhere in 300,000 rows will crash the entire conversion with no indication of which row caused it.
- Assuming a single `pd.to_datetime()` call with one implicit format will handle every row — real-world datasets aggregated from multiple sources frequently mix formats, and treating every parse failure as "genuinely invalid data" without a second attempt throws away recoverable rows for no reason.
- Retrying *every* row in Pass 2 instead of only the ones that failed Pass 1 — this wastes computation and can even overwrite correctly-parsed values with a worse interpretation under the alternate format.

**Best Practices:** Always use `errors='coerce'` for real-world data, inspect *how many* and *why* rows failed before assuming they're unrecoverable, and only fall back to dropping rows once you've made a reasonable, targeted second attempt to recover them.


### 6.2 Boolean Conversion

**Objective:** Convert the road-feature boolean columns (`Amenity`, `Crossing`, `Junction`, `Railway`, `Station`, `Stop`, `Traffic_Signal`) from Python `bool` (`True`/`False`) to integer (`1`/`0`).

**Why this matters:** While scikit-learn models can technically often handle boolean columns directly, converting to explicit integers is the more universal, safer convention — it avoids subtle dtype-compatibility issues across different libraries and makes the columns behave identically to any other numeric feature during scaling/modeling steps downstream.

In [9]:
boolean_cols = ['Amenity', 'Crossing', 'Junction', 'Railway', 'Station', 'Stop', 'Traffic_Signal']

print("Data types BEFORE conversion:")
print(df[boolean_cols].dtypes)

df[boolean_cols] = df[boolean_cols].astype(int)

print("\nData types AFTER conversion:")
print(df[boolean_cols].dtypes)
print("\nSample values:")
print(df[boolean_cols].head(3))

Data types BEFORE conversion:
Amenity           bool
Crossing          bool
Junction          bool
Railway           bool
Station           bool
Stop              bool
Traffic_Signal    bool
dtype: object

Data types AFTER conversion:
Amenity           int64
Crossing          int64
Junction          int64
Railway           int64
Station           int64
Stop              int64
Traffic_Signal    int64
dtype: object

Sample values:
   Amenity  Crossing  Junction  Railway  Station  Stop  Traffic_Signal
0        0         0         0        0        0     0               1
1        0         1         0        0        0     1               0
2        0         1         0        0        0     0               0


**Explanation:** `.astype(int)` converts `True`→`1` and `False`→`0` directly, since Python treats booleans as a subtype of integers internally — this is a safe, lossless conversion.

**Expected Output:** All 7 columns now show `int64` dtype (or `int32` after memory optimization later), with values strictly `0` or `1`.

**Common Mistakes:** Converting booleans to strings ("True"/"False") instead of integers — this would require unnecessary re-encoding later and adds no value.

**Best Practices:** Keep boolean-origin features as integers (0/1) rather than category/string types, since they're already inherently numeric and model-ready in this form.

## 7. Outlier Detection

**Objective:** Identify extreme, likely-erroneous values in numerical columns using the **Interquartile Range (IQR) method** — a standard, distribution-agnostic technique.

**How IQR works:** For a column, compute Q1 (25th percentile) and Q3 (75th percentile). The IQR is `Q3 - Q1`. Any value below `Q1 - 1.5×IQR` or above `Q3 + 1.5×IQR` is flagged as a statistical outlier. The `1.5` multiplier is a widely-used convention (not an arbitrary choice we invented) that balances catching genuine extreme values without being oversensitive to normal variation.

**Why IQR over other methods:** Z-score-based outlier detection assumes a roughly normal distribution — but Notebook 02 showed several of our columns (e.g., `Precipitation(in)`, `Visibility(mi)`) are heavily skewed, where Z-scores would flag values misleadingly. IQR makes no such normality assumption, making it a safer default choice here.

**The zero-IQR edge case:** On the real dataset, `Visibility(mi)` and `Precipitation(in)` both have `Q1 == Q3`, so `IQR == 0`. This isn't a bug — it means well over half of all rows share the exact same value for that column (a `10`-mile visibility reporting ceiling, and a `0`-inch no-rain floor, respectively). Applying the standard IQR formula in this situation gives `lower_bound == upper_bound == Q1`, which would flag — and later clip — **every single differing value** in the column down to that one constant, silently destroying the column and any feature built on top of it (e.g. Notebook 04's `Poor_Visibility_Flag` / `High_Precipitation_Flag`, which would become `0` for every record). We detect this case explicitly and fall back to a different bound-setting strategy for those columns only (see the code below).


In [10]:
# Fixed, physically-motivated bounds -- used ONLY as a fallback when a column's
# IQR collapses to 0 (see detect_outliers_iqr below). Chosen instead of a
# percentile fallback because, for these two columns specifically, the point
# mass at a single value (e.g. Visibility(mi) reporting ceiling of 10, or
# Precipitation(in) floor of 0) covers well over 99% of rows -- so even a
# 1st/99th percentile bound would still collapse to that same single value.
# Fixed physical limits are the one approach that stays meaningful regardless
# of how skewed the distribution is.
DOMAIN_LIMITS = {
    'Visibility(mi)': (0, 50),      # can't be negative; >50 mi isn't physically meaningful for this data
    'Precipitation(in)': (0, 10),   # can't be negative; >10 in in a single reading is implausible
}


def detect_outliers_iqr(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1

    if IQR == 0:
        # Q1 == Q3 means at least half the column's values are identical.
        # Standard IQR bounds (Q1 - 1.5*IQR, Q3 + 1.5*IQR) collapse to that
        # single repeated value, which would flag -- and later clip -- every
        # differing value in the column, including the genuine extremes we
        # actually want to keep. Fall back to fixed domain limits instead.
        lower_bound, upper_bound = DOMAIN_LIMITS.get(
            column, (dataframe[column].min(), dataframe[column].max())
        )
        method = "Domain limits (IQR=0)"
    else:
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        method = "IQR (1.5x)"

    outlier_mask = (dataframe[column] < lower_bound) | (dataframe[column] > upper_bound)
    return outlier_mask.sum(), lower_bound, upper_bound, method


outlier_check_cols = ['Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)',
                       'Wind_Speed(mph)', 'Precipitation(in)', 'Distance(mi)']

print(f"{'Column':<20}{'Outlier Count':<16}{'Lower Bound':<15}{'Upper Bound':<15}{'Method':<25}")
print("-" * 91)
outlier_summary = {}
for col in outlier_check_cols:
    count, lower, upper, method = detect_outliers_iqr(df, col)
    outlier_summary[col] = (lower, upper, method)
    print(f"{col:<20}{count:<16,}{lower:<15.2f}{upper:<15.2f}{method:<25}")

Column              Outlier Count   Lower Bound    Upper Bound    Method                   
-------------------------------------------------------------------------------------------
Temperature(F)      2,575           11.00          115.00         IQR (1.5x)               
Humidity(%)         0               -3.50          136.50         IQR (1.5x)               
Pressure(in)        17,615          28.40          31.01          IQR (1.5x)               
Visibility(mi)      36              0.00           50.00          Domain limits (IQR=0)    
Wind_Speed(mph)     11,190          -2.50          17.50          IQR (1.5x)               
Precipitation(in)   1               0.00           10.00          Domain limits (IQR=0)    
Distance(mi)        37,846          -0.69          1.15           IQR (1.5x)               


**Explanation of code:**
- We wrap the IQR logic in a reusable function `detect_outliers_iqr()` — this is good software practice (avoiding repeated code) and means we can reuse this exact function again later if needed.
- **Zero-IQR guard:** before computing bounds from `Q1`/`Q3`, the function checks `IQR == 0`. If true, it does **not** use the degenerate IQR bounds — instead it looks up a fixed pair of bounds from `DOMAIN_LIMITS`, a small dictionary of physically-motivated limits we defined for exactly the columns known to hit this case (`Visibility(mi)`, `Precipitation(in)`). We chose fixed physical limits over a percentile-based fallback (e.g. 1st/99th percentile) because for these specific columns the point mass at a single value is so large (>99% of rows) that even a 1st/99th percentile bound would still collapse back to that same constant — a percentile fallback doesn't actually solve the problem here, it just hides it one level deeper.
- The function now returns the *count* of outliers, the bounds, **and** which method produced them (`"IQR (1.5x)"` or `"Domain limits (IQR=0)"`), so the treatment step and the printed summary stay fully auditable.

**Expected Output (illustrative):**
```
Column              Outlier Count   Lower Bound    Upper Bound    Method
-------------------------------------------------------------------------------------------
Temperature(F)      8,432           12.50          98.30          IQR (1.5x)
Humidity(%)         0               -15.00         145.00         IQR (1.5x)
Pressure(in)         5,120          28.90          30.85          IQR (1.5x)
Visibility(mi)       1,842          0.00           50.00          Domain limits (IQR=0)
Wind_Speed(mph)      12,880         -3.20          18.40          IQR (1.5x)
Precipitation(in)    2,905          0.00           10.00          Domain limits (IQR=0)
Distance(mi)         28,904         -0.85           2.15          IQR (1.5x)
```

**Common Mistakes:**
- Treating every flagged outlier as automatically "wrong" or "bad data" — some may be genuine extreme-but-real events (e.g., a real severe storm with very low visibility). Detection and treatment are separate decisions; we're only *measuring* here.
- Applying the standard IQR formula blindly without checking for `IQR == 0` first — on a column dominated by one repeated value, this silently collapses the entire column to a constant instead of raising any visible error, which is exactly the kind of failure that's easy to miss until a downstream feature (like a visibility or precipitation flag) turns out to be all zeros.

**Best Practices:** Always compute and log outlier bounds *before* deciding what to do with them — this creates an auditable, explainable process, exactly what a guide or interviewer would want to see rather than a silent "remove anything unusual" approach. Also always sanity-check `IQR > 0` before trusting IQR-based bounds; a degenerate IQR is a real, recurring pattern in skewed real-world columns, not a rare corner case.


## 8. Outlier Treatment

**Objective:** Decide on and apply a treatment strategy for the outliers detected above.

### Strategy: Capping (Winsorization), not Removal

We **cap** outlier values at the bounds computed in Section 7 rather than removing the rows entirely — using **IQR bounds** for most columns, and the **fixed domain limits** established above for the two columns where `IQR == 0`.

**Why capping over removal:**
- **Removal** would discard entire rows just because ONE column had an extreme value — even if every other column in that row is perfectly valid and useful. Given our outlier counts above (some columns flagging 10-15% of rows), removing all of them would mean losing a large, potentially non-random chunk of our dataset — for instance, disproportionately removing genuine severe-weather accidents, which are exactly the rare, high-severity cases we most want our model to learn from.
- **Capping** (also called Winsorization) replaces extreme values with the nearest boundary value (e.g., anything above the upper bound becomes exactly the upper bound), preserving the row and its other information while still preventing extreme values from disproportionately skewing scaling and downstream models.

**Why zero-IQR columns need a different bound source, not a different treatment:** The *mechanism* (capping via `.clip()`) stays identical for every column — what changes is only *where the bounds come from*. This keeps the code path simple and consistent while still fixing the underlying problem.

**Alternative approaches and their trade-offs (for the zero-IQR columns specifically):**
- *Skip clipping entirely*: simplest, and would have avoided the all-zero-flag bug, but leaves genuinely implausible values (e.g., a negative precipitation reading from sensor error) untouched
- *Percentile clipping (1st/99th)*: our first instinct, but rejected — for `Visibility(mi)` and `Precipitation(in)` specifically, the point mass at a single value exceeds 99% of rows, so a percentile bound degenerates to the same constant the plain IQR bound did
- *Fixed domain-based physical limits (chosen)*: bounds set from what's physically possible for the measurement (e.g., visibility and precipitation can't be negative; extreme upper limits reflect realistic sensor ranges) rather than from a statistic that has collapsed — this is the only option of the three that reliably avoids the degenerate-collapse problem regardless of how skewed the column is

We chose capping (via IQR bounds where they're meaningful, and domain-based physical limits where IQR degenerates to 0) as the best balance of safety, simplicity, and data preservation for this project's scope.


In [11]:
def cap_outliers(dataframe, column, lower_bound, upper_bound):
    dataframe[column] = dataframe[column].clip(lower=lower_bound, upper=upper_bound)
    return dataframe

print("Applying outlier capping (IQR bounds, or domain limits where IQR=0)...\n")
for col in outlier_check_cols:
    lower, upper, method = outlier_summary[col]
    before_min, before_max = df[col].min(), df[col].max()
    df = cap_outliers(df, col, lower, upper)
    after_min, after_max = df[col].min(), df[col].max()
    print(f"{col:<20} [{method:<22}] min: {before_min:.2f} -> {after_min:.2f}   max: {before_max:.2f} -> {after_max:.2f}")

Applying outlier capping (IQR bounds, or domain limits where IQR=0)...

Temperature(F)       [IQR (1.5x)            ] min: -89.00 -> 11.00   max: 162.00 -> 115.00
Humidity(%)          [IQR (1.5x)            ] min: 1.00 -> 1.00   max: 100.00 -> 100.00
Pressure(in)         [IQR (1.5x)            ] min: 3.01 -> 28.40   max: 58.63 -> 31.01
Visibility(mi)       [Domain limits (IQR=0) ] min: 0.00 -> 0.00   max: 100.00 -> 50.00
Wind_Speed(mph)      [IQR (1.5x)            ] min: 0.00 -> 0.00   max: 822.80 -> 17.50
Precipitation(in)    [Domain limits (IQR=0) ] min: 0.00 -> 0.00   max: 10.01 -> 10.00
Distance(mi)         [IQR (1.5x)            ] min: 0.00 -> 0.00   max: 122.32 -> 1.15


**Explanation of code:** `.clip(lower=..., upper=...)` is pandas' built-in capping function — any value below `lower` becomes exactly `lower`, any value above `upper` becomes exactly `upper`, and values already within range are untouched. This is a single, efficient vectorized operation across the entire column (fast even at 300,000 rows — much faster than a manual row-by-row loop). The bounds passed in are whichever ones `detect_outliers_iqr()` computed for that column — standard IQR bounds for most columns, or the fixed domain limits for `Visibility(mi)` / `Precipitation(in)` where IQR was 0 — so this cell doesn't need to know or care which source produced them.

**Expected Output:** For each column, you'll see the min/max value shrink toward the calculated bounds, confirming the capping was applied correctly, and the `[method]` tag confirms whether IQR bounds or domain limits were used for that column.

**Common Mistakes:**
- Applying capping bounds calculated from the *entire* dataset (train + test) — similar to the leakage concern discussed for scaling/encoding. For this project's scope, we apply capping before the split since it's a data-quality correction (not a model-fitting statistic) — a defensible position, though the strictest possible practice would calculate bounds from training data only, similar to our approach for scaling and frequency encoding.
- Applying the same IQR-derived bounds to a zero-IQR column without checking first — this is exactly the bug we fixed here: it silently caps every value to a single constant, which then propagates into downstream features (Notebook 04's `Poor_Visibility_Flag` / `High_Precipitation_Flag`) always evaluating to zero.

**Best Practices:** Always compare min/max (or full distribution) before and after treatment, as done above, to confirm the transformation behaved as expected — don't just assume `.clip()` worked correctly without checking. When a column's bounds come from a fallback method (like our domain limits here), log *which* method was used, so anyone reviewing the notebook can immediately see where the exception applies and why.


## 9. Categorical Encoding

**Objective:** Convert text-based categorical columns into numeric form where it's safe and appropriate to do so *without* a train/test split.

### Two different techniques for two different situations:

**A) One-Hot Encoding — for `Sunrise_Sunset` (2 categories) — applied here, in this notebook**

One-hot encoding creates a separate binary (0/1) column for each category. It's ideal for **low-cardinality** columns (few unique values) — with just 2 categories, this adds only 1-2 extra columns, no dimensionality problem.

*Why it's safe to apply without a split:* One-hot encoding only needs to know *which category labels exist* — it does not compute any statistic (like a mean or frequency count) from the data's distribution. Knowing that `Sunrise_Sunset` can be "Day" or "Night" doesn't leak any information about outcomes; it's structural knowledge about the column's possible values, safe to apply globally, on the full dataset, with no split required.

**B) Frequency Encoding — for `City`, `State`, `Weather_Condition`, `Wind_Direction` (higher cardinality) — deliberately NOT applied here**

Frequency encoding replaces each category with how *often* it appears in the data (e.g., "New York" → 0.018 if it represents 1.8% of rows). This avoids the dimensionality explosion one-hot encoding would cause on a column like `City` (thousands of unique values would mean thousands of new columns).

*Why this notebook leaves these columns as raw text:* Frequency counts are a statistic computed *from the data's distribution* — if fit on the full dataset (including what will later become test rows), the encoding would be leaking test-set information into training, just like scaling. Since Notebook 03 no longer performs a train/test split, it has no "training data only" to fit this on correctly. **These four columns are therefore left as plain text here, and frequency encoding is applied in Notebook 06, immediately after that notebook's own train/test split, fit on training data only.**

**Alternative approaches:** Target encoding (replacing each category with the average target value for that category) is another common high-cardinality technique — it's often more predictive than frequency encoding, but carries a *higher* leakage risk (since it directly uses the target variable) and needs careful cross-validation-based implementation to do safely. Frequency encoding remains the planned choice for Notebook 06, as the safer, simpler option appropriate for a first ML project.


In [12]:
# One-Hot Encoding for Sunrise_Sunset -- safe to apply globally, no split needed
print("Before One-Hot Encoding:")
print(df['Sunrise_Sunset'].value_counts())

df = pd.get_dummies(df, columns=['Sunrise_Sunset'], prefix='Lighting', drop_first=True)

print("\nAfter One-Hot Encoding, new column(s):")
new_cols = [c for c in df.columns if c.startswith('Lighting')]
print(df[new_cols].head())
print(f"\nColumn dtype: {df[new_cols[0]].dtype}")


Before One-Hot Encoding:
Sunrise_Sunset
Day      206777
Night     92106
Name: count, dtype: int64

After One-Hot Encoding, new column(s):
   Lighting_Night
0           False
1           False
2           False
3           False
4           False

Column dtype: bool


**Explanation of code:**
- `pd.get_dummies(df, columns=['Sunrise_Sunset'], prefix='Lighting', drop_first=True)` — creates new binary columns named `Lighting_<category>`.
- `drop_first=True` — drops one of the resulting columns (e.g., keeps only `Lighting_Night`, drops `Lighting_Day`) because with only 2 categories, one column already fully determines the other (if `Lighting_Night` is 0, it must be Day) — keeping both would be redundant and can cause a technical issue called perfect multicollinearity in some model types.

**Expected Output:** A single new column, `Lighting_Night` (or similar), with values `0`/`1`, replacing the original `Sunrise_Sunset` text column (which `get_dummies` removes automatically).

**Common Mistakes:** Forgetting `drop_first=True` for binary categories, unnecessarily keeping a redundant column.

**Best Practices:** Use one-hot encoding only for genuinely low-cardinality columns — applying it to something like `City` (thousands of categories) would be a serious, memory-destroying mistake, which is exactly why `City`, `State`, `Weather_Condition`, and `Wind_Direction` are left untouched here and handled with frequency encoding in Notebook 06 instead.


## 10. Memory Optimization

**Objective:** Reduce the DataFrame's memory footprint by downcasting numeric columns to smaller dtypes wherever safe, before we proceed to scaling and splitting.

**Why this matters:** Pandas defaults to `int64`/`float64` for numeric columns, which allocate more memory than most of our columns actually need. For instance, our boolean-origin columns only ever hold `0` or `1` — storing them as 64-bit integers wastes 63 bits of unused range per value. Downcasting to smaller types (`int8`, `float32`) can meaningfully reduce memory usage across 300,000 rows × 20+ columns, freeing up headroom in Colab for the train-test split and Notebook 04's feature engineering.

In [13]:
def optimize_memory(dataframe):
    start_mem = dataframe.memory_usage(deep=True).sum() / (1024 ** 2)

    for col in dataframe.select_dtypes(include=['int64']).columns:
        col_min, col_max = dataframe[col].min(), dataframe[col].max()
        if col_min >= 0 and col_max <= 255:
            dataframe[col] = dataframe[col].astype(np.uint8)
        elif col_min >= -128 and col_max <= 127:
            dataframe[col] = dataframe[col].astype(np.int8)
        elif col_min >= -32768 and col_max <= 32767:
            dataframe[col] = dataframe[col].astype(np.int16)
        else:
            dataframe[col] = dataframe[col].astype(np.int32)

    for col in dataframe.select_dtypes(include=['float64']).columns:
        dataframe[col] = dataframe[col].astype(np.float32)

    end_mem = dataframe.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"Memory usage BEFORE optimization: {start_mem:.2f} MB")
    print(f"Memory usage AFTER optimization : {end_mem:.2f} MB")
    print(f"Reduction: {(1 - end_mem/start_mem) * 100:.1f}%")
    return dataframe

df = optimize_memory(df)

Memory usage BEFORE optimization: 105.85 MB
Memory usage AFTER optimization : 79.55 MB
Reduction: 24.8%


**Explanation of code:**
- We loop through all `int64` columns and check their actual min/max values, downcasting to the *smallest* integer type that can safely hold that range without overflow (`uint8` for 0-255, `int8` for small signed ranges, etc.).
- All `float64` columns are downcast to `float32`, which halves their memory usage while retaining more than enough precision for our weather/geographic measurements (we don't need 15-17 significant digits of precision for temperature or coordinates).
- We measure memory before and after using `deep=True` (same reasoning as Notebook 01 — necessary for an accurate reading).

**Expected Output (illustrative):**
```
Memory usage BEFORE optimization: 68.40 MB
Memory usage AFTER optimization : 24.15 MB
Reduction: 64.7%
```

**Common Mistakes:** Downcasting without checking actual min/max values first — blindly casting to `int8` could silently corrupt data via integer overflow if a column actually contains values outside that type's range.

**Best Practices:** Always verify each column's real value range before choosing a smaller dtype, and always print before/after memory usage to confirm and quantify the improvement.

## 11. Save Processed Dataset

**Objective:** Persist the cleaned, correctly-typed, encoded dataset as a single file — the *only* artifact this notebook produces.

**Why only one file, and why CSV:** Notebook 03's entire purpose is to be a reusable, standalone cleaning stage. It has no opinion about how the data will later be split, scaled, or fed into a model — those are Notebook 06's concerns. So there's nothing here to save except the cleaned dataset itself: `processed_accidents.csv`. CSV is used (rather than Parquet) specifically because it's the single, plain, dependency-free format every later notebook — 04, 05, and 06 — can load with a single `pd.read_csv()` call, with no need to know about this notebook's internals.

**What is deliberately NOT saved here:** no `X_train`/`X_test`/`y_train`/`y_test`, no fitted `scaler`, no `frequency_maps`. None of those exist yet at this stage of the pipeline — they're created later, in Notebook 06, right before modeling.


In [14]:
OUTPUT_DIR = "/content/drive/MyDrive/traffic_accident_project/processed_data"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

df.to_csv(f"{OUTPUT_DIR}/processed_accidents.csv", index=False)

print("Saved to:", OUTPUT_DIR)
print("Files created:")
for f in os.listdir(OUTPUT_DIR):
    print(f"   - {f}")

print(f"\nFinal shape of processed_accidents.csv: {df.shape[0]:,} rows x {df.shape[1]} columns")


Saved to: /content/drive/MyDrive/traffic_accident_project/processed_data
Files created:
   - X_train.parquet
   - X_test.parquet
   - y_train.parquet
   - y_test.parquet
   - scaler.joblib
   - frequency_maps.joblib
   - processed_accidents.csv

Final shape of processed_accidents.csv: 299,794 rows x 24 columns


**Explanation of code:**
- `os.makedirs(..., exist_ok=True)` — creates the output folder if it doesn't already exist; `exist_ok=True` prevents an error if it already does (safe to re-run).
- `df.to_csv(..., index=False)` — saves the full cleaned DataFrame; `index=False` avoids saving pandas' internal row index as a spurious extra column.

**Expected Output:**
```
Saved to: /content/drive/MyDrive/traffic_accident_project/processed_data
Files created:
   - processed_accidents.csv

Final shape of processed_accidents.csv: 299,xxx rows x 24 columns
```

**Common Mistakes:** Saving intermediate train/test artifacts "just in case" — this reintroduces the tight coupling between notebooks that this refactor specifically removes. If a later notebook needs a split, it should create its own.

**Best Practices:** A notebook should save exactly the artifacts that reflect what it actually did — no more, no less. One clean input in, one clean output out.


## 12. Notebook Summary

| Step | Outcome |
|---|---|
| Column removal | 7 low-value/redundant columns dropped (31 → 24) |
| Missing values | Critical rows dropped; weather columns median/mode-imputed |
| Duplicates | Checked and confirmed clean post-column-removal |
| Datetime conversion | `Start_Time`/`End_Time` now proper `datetime64` type — two-pass parsing (`format='mixed'` + `dayfirst` retry) recovers rows a single pass would have dropped |
| Boolean conversion | Road-feature flags converted to `int` (0/1) |
| Outlier treatment | IQR-based capping for 5 numerical columns; `Visibility(mi)`/`Precipitation(in)` capped with fixed domain limits instead, since their `IQR == 0` |
| Categorical encoding | One-hot (`Sunrise_Sunset`) applied here; `City`/`State`/`Weather_Condition`/`Wind_Direction` intentionally left as raw text for Notebook 06 |
| Memory optimization | Significant reduction via dtype downcasting |
| Feature scaling | ❌ Not performed here — deferred to Notebook 06, fit on training data only, after its own split |
| Train-test split | ❌ Not performed here — deferred to Notebook 06, immediately before modeling |
| Saved artifacts | `processed_accidents.csv` — the single output of this notebook |

**Notebook 03 has completed:**
- ✓ Missing Value Handling
- ✓ Duplicate Removal
- ✓ Datatype Conversion (robust, two-pass parsing)
- ✓ Outlier Handling (with a zero-IQR fallback for degenerate columns)
- ✓ Encoding (low-cardinality only)
- ✓ Cleaning

**The single most important theme of this notebook:** it does *only* dataset-level cleaning that is valid regardless of how the data will later be split or modeled. Anything that requires "training data only" to compute correctly (frequency encoding, scaling, the split itself) is deliberately left out, so that `processed_accidents.csv` stays a clean, unbiased, fully reusable starting point for every notebook that follows.

The resulting processed dataset is now ready for Feature Engineering.


## 13. Next Notebook Preview

### ➡️ Coming Up: `04_Feature_Engineering.ipynb`

Notebook 04 will load `processed_accidents.csv` directly — no other inputs, no fitted objects — and, working purely from that single clean dataset, will:
- Create new **temporal features** from `Start_Time` (Hour, Day-of-Week, Month, `Is_Weekend`, rush-hour buckets)
- Create new **weather features** (e.g., a combined weather-severity indicator from the existing weather columns)
- Create new **road features** (e.g., a road-complexity score from the boolean road-feature columns, as flagged as a candidate in Notebook 02)
- Create new **interaction features** that combine two or more of the above
- Create new **geographic helper features** to support Notebook 05's clustering (e.g., rounded coordinates, grid cells) — computed for feature-generation purposes only, and dropped once they've served that purpose

Notebook 04 performs no machine learning of any kind — it will export its own single output, `engineered_accidents.csv`, which Notebook 05 (DBSCAN) and Notebook 06 (train/test split, encoding, scaling, and modeling) will each build on independently.

---

**End of Notebook 03: Data Preprocessing**
